# 01a - Cleaning with PySpark (Cleaning rules v1)
**HMDA 2024 project - Group 7.** Read the raw Parquet, apply the scope filter, clean with *Cleaning rules v1*, check the numbers and save `hmda_clean_v1_spark`.

Same rules as the Pandas version (`01b`): invalid values are **flagged and set to null**, records are not dropped. The last cells **compare the result with the Pandas version**, which is the first output of the correctness comparison.

> Note: Spark 4 runs in ANSI mode, so casting uses `try_cast` and division uses `try_divide`, which turn bad values into null instead of raising an error.

## Step 1. Mount Drive, copy the raw data to Colab, start Spark

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil
DRIVE_DIR     = "/content/drive/MyDrive/hmda"
PARQUET_DRIVE = f"{DRIVE_DIR}/hmda_parquet"
PARQUET_LOCAL = "/content/hmda_parquet"
RESULTS_DIR   = f"{DRIVE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(PARQUET_DRIVE), "hmda_parquet not found on Drive - run notebook 00 or check the shortcut"
if not os.path.exists(PARQUET_LOCAL):
    !cp -r "{PARQUET_DRIVE}" "{PARQUET_LOCAL}"
print("OK")

Mounted at /content/drive
OK


In [2]:
!pip install -q pyspark==4.0.5   # same version as requirements.txt
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]")
         .config("spark.driver.memory", "8g")
         .appName("hmda_clean_pyspark").getOrCreate())
print("Spark", spark.version)

Spark 4.0.5


## Step 2. Rule configuration (identical to the Pandas version)

In [3]:
SCOPE_VALUES = {                      # column -> allowed values (strings)
    "action_taken": ["1", "2", "3"],
    "occupancy_type": ["1"],
    "loan_purpose": ["1"],
    "lien_status": ["1"],
    "derived_dwelling_category": ["Single Family (1-4 Units):Site-Built"],
    "reverse_mortgage": ["2"],
    "open_end_line_of_credit": ["2"],
    "business_or_commercial_purpose": ["2"],
}
NUMERIC_COLS = ["loan_amount", "income", "combined_loan_to_value_ratio", "property_value", "loan_term"]
KEEP_COLS = ["action_taken", "state_code", "derived_msa_md", "loan_type", "lei", "debt_to_income_ratio",
             "derived_race", "derived_ethnicity", "derived_sex", "applicant_age"] + NUMERIC_COLS
INT32_MAX = 2147483647

REGIONS = {
    "Northeast": "CT ME MA NH RI VT NJ NY PA",
    "Midwest": "IL IN MI OH WI IA KS MN MO NE ND SD",
    "South": "DE DC FL GA MD NC SC VA WV AL KY MS TN AR LA OK TX",
    "West": "AZ CO ID MT NV NM UT WY AK CA HI OR WA",
    "Territory": "PR VI GU AS MP",
}

## Step 3. Read the data and apply the scope filter (record counts after each filter)

In [4]:
raw = spark.read.parquet(PARQUET_LOCAL)
steps = {"total_records": raw.count()}

scope = raw
for col, vals in SCOPE_VALUES.items():
    scope = scope.filter(F.col(col).isin(vals))
    steps[f"after_filter_{col}"] = scope.count()
    print(f"{col:34s} {steps[f'after_filter_{col}']:>12,}")

sdf = scope.select(*KEEP_COLS).cache()      # keep in RAM because it is reused several times
print("Records in scope:", f"{sdf.count():,}")

action_taken                          8,661,779
occupancy_type                        7,792,568
loan_purpose                          3,811,298
lien_status                           3,641,962
derived_dwelling_category             3,237,835
reverse_mortgage                      3,168,105
open_end_line_of_credit               3,145,390
business_or_commercial_purpose        3,140,857
Records in scope: 3,140,857


## Step 4. Cleaning function `clean_spark()`
Every rule R3-R11 lives in this function. Rule R12 (duplicates) is in the optional cell at the end of the notebook.

In [5]:
def num(c):
    """Cast strings to numbers; 'NA', 'Exempt' or unreadable values become null (never 0)."""
    return F.expr(f"try_cast(`{c}` as double)")

def band(col, edges, labels):
    """Half-open bands [a, b); null or below the lowest edge becomes 'Missing'."""
    e = F.when(F.col(col).isNull() | (F.col(col) < edges[0]), "Missing")
    for hi, lab in zip(edges[1:-1], labels[:-1]):
        e = e.when(F.col(col) < hi, lab)
    return e.otherwise(labels[-1])

def dti_band_expr():
    d = F.trim(F.col("debt_to_income_ratio"))
    n = F.expr("try_cast(trim(debt_to_income_ratio) as double)")
    return (F.when(d == "<20%", "<20").when(d == "20%-<30%", "20-<30").when(d == "30%-<36%", "30-<36")
             .when(d == "50%-60%", "50-60").when(d == ">60%", ">60")
             .when(n.isNull(), "Missing")                       # 'NA', 'Exempt', unreadable
             .when(n < 20, "<20").when(n < 30, "20-<30").when(n < 36, "30-<36").when(n < 40, "36-<40")
             .when(n < 45, "40-<45").when(n < 50, "45-<50").when(n <= 60, "50-60").otherwise(">60"))

def region_expr():
    e = F.lit("Unknown")
    for region, states in REGIONS.items():
        e = F.when(F.col("state_code").isin(states.split()), region).otherwise(e)
    return e

def clean_spark(sdf):
    d = sdf
    for c in NUMERIC_COLS:                                                    # R3
        d = d.withColumn(c, num(c))
    flag = lambda cond: F.when(cond, 1).otherwise(0)                          # null -> 0
    d = (d.withColumn("income_invalid", flag(F.col("income") <= 0))                                       # R4
           .withColumn("cltv_invalid", flag((F.col("combined_loan_to_value_ratio") > 200) |
                                            (F.col("combined_loan_to_value_ratio") < 5)))                # R5
           .withColumn("property_value_invalid", flag(F.col("property_value") == INT32_MAX))              # R6
           .withColumn("loan_term_invalid", flag((F.col("loan_term") < 12) | (F.col("loan_term") > 480))))  # R7
    for f, c in [("income_invalid", "income"), ("cltv_invalid", "combined_loan_to_value_ratio"),
                 ("property_value_invalid", "property_value"), ("loan_term_invalid", "loan_term")]:
        d = d.withColumn(c, F.when(F.col(f) == 1, F.lit(None).cast("double")).otherwise(F.col(c)))

    non_metro = F.col("derived_msa_md") == "99999"                            # R9
    d = (d.withColumn("action_taken", F.col("action_taken").cast("int"))
           .withColumn("denied", F.when(F.col("action_taken") == 3, 1).otherwise(0))                       # R2
           .withColumn("census_region", region_expr())
           .withColumn("metro_flag", F.when(non_metro, 0).otherwise(1))
           .withColumn("msa_md_clean", F.when(non_metro, "Non-metro").otherwise(F.col("derived_msa_md")))
           .withColumn("loan_to_income", F.expr("try_divide(loan_amount, income * 1000)"))                  # R10
           .withColumn("dti_band", dti_band_expr())                                                        # R8
           .withColumn("income_band", band("income", [0, 50, 75, 100, 150, 250, float("inf")],
                                           ["<50", "50-<75", "75-<100", "100-<150", "150-<250", "250+"]))
           .withColumn("loan_amount_band", band("loan_amount", [0, 150e3, 250e3, 350e3, 500e3, 1e6, float("inf")],
                                                ["<150k", "150-<250k", "250-<350k", "350-<500k", "500k-<1M", "1M+"]))
           .withColumn("cltv_band", band("combined_loan_to_value_ratio", [0, 60, 80, 90, 97, float("inf")],
                                         ["<60", "60-<80", "80-<90", "90-<97", "97+"]))
           .withColumn("income_gt5000", flag(F.col("income") > 5000))                                      # R11
           .withColumn("lti_gt10", flag(F.col("loan_to_income") > 10)))                                    # R10

    return d.select(
        "action_taken", "denied", "state_code", "census_region", "derived_msa_md", "msa_md_clean", "metro_flag",
        "loan_type", "lei", "loan_amount", "income", "combined_loan_to_value_ratio", "property_value", "loan_term",
        "loan_to_income", "dti_band", "income_band", "loan_amount_band", "cltv_band",
        "income_invalid", "cltv_invalid", "property_value_invalid", "loan_term_invalid", "income_gt5000", "lti_gt10",
        F.col("derived_race").alias("val_race"), F.col("derived_ethnicity").alias("val_ethnicity"),
        F.col("derived_sex").alias("val_sex"), F.col("applicant_age").alias("val_age"))
print("clean_spark() defined")

clean_spark() defined


## Step 5. Run the cleaning and measure the check metrics
Spark is lazy: the computation only happens when `count()` or `collect()` is called. This step takes a few minutes.

In [6]:
t0 = time.perf_counter()
out = clean_spark(sdf).cache()

special = (sdf.agg(*[F.sum(F.col(c).isin("NA", "Exempt").cast("int")).alias(c) for c in NUMERIC_COLS])
              .collect()[0].asDict())
conv_null = (sdf.select(*[num(c).alias(c) for c in NUMERIC_COLS])
                .agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in NUMERIC_COLS])
                .collect()[0].asDict())
m = out.agg(F.count("*").alias("rows"), F.sum("denied").alias("denied"),
            F.sum("income_invalid").alias("income_invalid"), F.sum("cltv_invalid").alias("cltv_invalid"),
            F.sum("property_value_invalid").alias("property_value_invalid"),
            F.sum("loan_term_invalid").alias("loan_term_invalid"),
            F.sum("income_gt5000").alias("income_gt5000"), F.sum("lti_gt10").alias("lti_gt10")).collect()[0].asDict()
dti_counts = {r["dti_band"]: r["count"] for r in out.groupBy("dti_band").count().collect()}

log = {"rows_in_scope": m["rows"], "rows_out": m["rows"], "denied_rows": m["denied"],
       "scope_step_counts": steps,
       "special_NA_or_Exempt": special,
       "unparseable_other_than_NA_Exempt": {c: conv_null[c] - special[c] for c in NUMERIC_COLS},
       "invalid_flagged_and_nulled": {k: m[k] for k in ["income_invalid", "cltv_invalid",
                                                        "property_value_invalid", "loan_term_invalid"]},
       "kept_but_flagged": {"income_gt5000": m["income_gt5000"], "lti_gt10": m["lti_gt10"]},
       "dti_band_counts": dti_counts,
       "denied_rate": round(m["denied"] / m["rows"], 6)}
print(f"Xong sau {time.perf_counter()-t0:.0f}s")

Xong sau 75s


## Step 6. Check the numbers against the earlier measurements
Every line must show **OK**. If any line shows **MISMATCH**, stop and do not save.

In [7]:
expected = {
    "total_records":                    (steps["total_records"], 12259199),
    "records with action_taken 1-3":    (steps["after_filter_action_taken"], 8661779),
    "rows in scope":                    (log["rows_in_scope"], 3140857),
    "denied rows":                      (log["denied_rows"], 276604),
    "NA/Exempt income":                 (log["special_NA_or_Exempt"]["income"], 21366),
    "NA/Exempt CLTV":                   (log["special_NA_or_Exempt"]["combined_loan_to_value_ratio"], 96968),
    "NA/Exempt property_value":         (log["special_NA_or_Exempt"]["property_value"], 29439),
    "NA/Exempt loan_term":              (log["special_NA_or_Exempt"]["loan_term"], 6160),
    "income <= 0 (invalid)":            (log["invalid_flagged_and_nulled"]["income_invalid"], 16942),
    "CLTV >200 or <5 (invalid)":        (log["invalid_flagged_and_nulled"]["cltv_invalid"], 371 + 288),
    "property_value = 2^31-1 (invalid)":(log["invalid_flagged_and_nulled"]["property_value_invalid"], 2),
    "loan_term <12 or >480 (invalid)":  (log["invalid_flagged_and_nulled"]["loan_term_invalid"], 1001 + 4),
    "income > 5000 (flag)":             (log["kept_but_flagged"]["income_gt5000"], 857),
    "loan_to_income > 10 (flag)":       (log["kept_but_flagged"]["lti_gt10"], 11937),
    "DTI band <20":    (dti_counts.get("<20", 0), 113535),
    "DTI band 20-<30": (dti_counts.get("20-<30", 0), 390684),
    "DTI band 30-<36": (dti_counts.get("30-<36", 0), 449579),
    "DTI band 36-<40": (dti_counts.get("36-<40", 0), 401994),
    "DTI band 40-<45": (dti_counts.get("40-<45", 0), 628198),
    "DTI band 45-<50": (dti_counts.get("45-<50", 0), 710151),
    "DTI band 50-60":  (dti_counts.get("50-60", 0), 338631),
    "DTI band >60":    (dti_counts.get(">60", 0), 82947),
    "DTI band Missing":(dti_counts.get("Missing", 0), 25138),
    "unparseable (should be 0)":        (sum(log["unparseable_other_than_NA_Exempt"].values()), 0),
}
bad = 0
for name, (got, exp) in expected.items():
    ok = got == exp
    bad += (not ok)
    print(f"{'OK       ' if ok else 'MISMATCH '} {name:36s} got={got:>10,}  expected={exp:>10,}")
print("\n=> ALL MATCH" if bad == 0 else f"\n=> {bad} metrics MISMATCH - results not saved")

OK        total_records                        got=12,259,199  expected=12,259,199
OK        records with action_taken 1-3        got= 8,661,779  expected= 8,661,779
OK        rows in scope                        got= 3,140,857  expected= 3,140,857
OK        denied rows                          got=   276,604  expected=   276,604
OK        NA/Exempt income                     got=    21,366  expected=    21,366
OK        NA/Exempt CLTV                       got=    96,968  expected=    96,968
OK        NA/Exempt property_value             got=    29,439  expected=    29,439
OK        NA/Exempt loan_term                  got=     6,160  expected=     6,160
OK        income <= 0 (invalid)                got=    16,942  expected=    16,942
OK        CLTV >200 or <5 (invalid)            got=       659  expected=       659
OK        property_value = 2^31-1 (invalid)    got=         2  expected=         2
OK        loan_term <12 or >480 (invalid)      got=     1,005  expected=     1,005
OK  

## Step 7. Save `hmda_clean_v1_spark` and the log
Run only when Step 6 reports **ALL MATCH**. Write to the local disk first, then copy to Drive (writing directly to Drive is very slow).

In [8]:
OUT_LOCAL = "/content/hmda_clean_v1_spark"
OUT_DRIVE = f"{DRIVE_DIR}/hmda_clean_v1_spark"

out.coalesce(1).write.mode("overwrite").parquet(OUT_LOCAL)
with open(f"{OUT_LOCAL}/cleaning_log_spark.json", "w") as fh:
    json.dump(log, fh, indent=2, default=int)

if os.path.exists(OUT_DRIVE):
    shutil.rmtree(OUT_DRIVE)               # this notebook's own output folder, overwrite the old version
shutil.copytree(OUT_LOCAL, OUT_DRIVE)
shutil.copy(f"{OUT_LOCAL}/cleaning_log_spark.json", f"{RESULTS_DIR}/cleaning_log_spark.json")
print(os.listdir(OUT_DRIVE))

['part-00000-02e71141-e2c5-4a74-8932-0d8713e4e727-c000.snappy.parquet', '_SUCCESS', '.part-00000-02e71141-e2c5-4a74-8932-0d8713e4e727-c000.snappy.parquet.crc', '._SUCCESS.crc', 'cleaning_log_spark.json']


## Step 8. (Optional) Compare with the Pandas version
First output of the **correctness comparison**: same input, same rules, the two tools must give the same result. Counts must be **exactly equal**; means are compared with a relative tolerance of 1e-9.
Can run once `hmda_clean_v1/hmda_clean_v1.parquet` exists (the output of notebook `01b`).

In [9]:
PANDAS_OUT = f"{DRIVE_DIR}/hmda_clean_v1/hmda_clean_v1.parquet"
if not os.path.exists(PANDAS_OUT):
    print("Pandas result not found - skipping this cell")
else:
    pdf = spark.read.parquet(PANDAS_OUT)
    SUM_COLS = ["denied", "income_invalid", "cltv_invalid", "property_value_invalid", "loan_term_invalid",
                "income_gt5000", "lti_gt10", "metro_flag"]
    MEAN_COLS = ["loan_amount", "income", "combined_loan_to_value_ratio", "property_value", "loan_term", "loan_to_income"]
    BAND_COLS = ["dti_band", "income_band", "loan_amount_band", "cltv_band", "census_region"]

    def summarize(x):
        agg = x.agg(F.count("*").alias("rows"),
                    *[F.sum(c).alias("sum_" + c) for c in SUM_COLS],
                    *[F.count(c).alias("nonnull_" + c) for c in MEAN_COLS],
                    *[F.avg(c).alias("mean_" + c) for c in MEAN_COLS]).collect()[0].asDict()
        bands = {c: {r[c]: r["count"] for r in x.groupBy(c).count().collect()} for c in BAND_COLS}
        return agg, bands

    a_spark, b_spark = summarize(spark.read.parquet(OUT_LOCAL + "/*.parquet"))   # read only the .parquet files, skip the .json log
    a_pd, b_pd = summarize(pdf)
    bad = 0
    for k in a_spark:
        s, p = a_spark[k], a_pd[k]
        ok = (s == p) if not k.startswith("mean_") else abs(s - p) <= 1e-9 * max(1.0, abs(p))
        bad += (not ok)
        print(f"{'OK       ' if ok else 'MISMATCH '} {k:40s} spark={s}  pandas={p}")
    for c in BAND_COLS:
        ok = b_spark[c] == b_pd[c]
        bad += (not ok)
        print(f"{'OK       ' if ok else 'MISMATCH '} bands:{c}")
    print("\n=> SPARK = PANDAS" if bad == 0 else f"\n=> {bad} metrics MISMATCH - find the cause")

OK        rows                                     spark=3140857  pandas=3140857
OK        sum_denied                               spark=276604  pandas=276604
OK        sum_income_invalid                       spark=16942  pandas=16942
OK        sum_cltv_invalid                         spark=659  pandas=659
OK        sum_property_value_invalid               spark=2  pandas=2
OK        sum_loan_term_invalid                    spark=1005  pandas=1005
OK        sum_income_gt5000                        spark=857  pandas=857
OK        sum_lti_gt10                             spark=11937  pandas=11937
OK        sum_metro_flag                           spark=2803339  pandas=2803339
OK        nonnull_loan_amount                      spark=3140857  pandas=3140857
OK        nonnull_income                           spark=3102549  pandas=3102549
OK        nonnull_combined_loan_to_value_ratio     spark=3043230  pandas=3043230
OK        nonnull_property_value                   spark=3111416  pandas

## Step 9. (Optional) Rule R12: count duplicate records over all 99 columns
The Pandas version counts duplicates only over the kept columns. Here they are counted over **all 99 columns** of the in-scope records. Duplicates are **counted, not removed**. This cell is heavy (reads all 99 columns) and can take a few minutes.

In [10]:
scope_all = raw
for col, vals in SCOPE_VALUES.items():
    scope_all = scope_all.filter(F.col(col).isin(vals))
n_all = scope_all.count()
n_distinct = scope_all.dropDuplicates().count()
log["exact_duplicate_rows_all_99_columns"] = n_all - n_distinct
print(f"Records in scope: {n_all:,} | distinct: {n_distinct:,} | duplicates: {n_all - n_distinct:,}")
with open(f"{RESULTS_DIR}/cleaning_log_spark.json", "w") as fh:
    json.dump(log, fh, indent=2, default=int)

Records in scope: 3,140,857 | distinct: 3,140,667 | duplicates: 190
